<a href="https://colab.research.google.com/github/YUFEIFUT/Demos/blob/main/%E4%BA%BA%E8%84%B8%E8%AF%86%E5%88%ABDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 安装 InsightFace 和相关依赖
InsightFace 是目前性能最强的人脸识别开源工具包包之一。我们首先需要安装它以及 ONNX Runtime（用于推理驱动）。

In [ ]:
!pip install -U "numpy<2" insightface onnxruntime-gpu opencv-python

### 初始化 InsightFace 模型
我们将使用 InsightFace 提供的 `buffalo_l` 模型库，这是一个包含人脸检测、识别、对齐等多个任务的大型模型库。

In [ ]:
import cv2
import numpy as np
from insightface.app import FaceAnalysis
from google.colab.patches import cv2_imshow

# 修复 NumPy 2.x/1.24+ 的兼容性问题，因为 insightface 内部使用了 np.int
if not hasattr(np, 'int'):
    np.int = int

# 初始化人脸分析应用
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

# 读取一张测试图片
img_path = '/content/test1.jpg'
img = cv2.imread(img_path)

if img is not None:
    # 进行人脸检测和特征提取
    faces = app.get(img)

    # 绘制识别结果
    res_img = app.draw_on(img, faces)

    print(f"检测到 {len(faces)} 张人脸")
    cv2_imshow(res_img)
else:
    print("请确保上传了测试图片 /content/test1.jpg 并修改了路径。")

### 测试另一张图片 (test2.jpg)
我们现在尝试对 `/content/test2.jpg` 进行同样的人脸分析。

In [ ]:
img_path2 = '/content/41813孙佳美.jpg'
img2 = cv2.imread(img_path2)

if img2 is not None:
    # 进行人脸检测和特征提取
    faces2 = app.get(img2)

    # 绘制识别结果
    res_img2 = app.draw_on(img2, faces2)

    print(f"检测到 {len(faces2)} 张人脸")
    cv2_imshow(res_img2)
else:
    print("请确保上传了测试图片 /content/test2.jpg 并修改了路径。")

### 进阶：构建简单的人脸识别系统
下面的代码演示了如何提取特征向量并进行 1:N 的搜索比对。

In [ ]:
# 1. 模拟一个简单的底库 (Gallery)
face_db = {}

def register_face(name, image_path):
    img = cv2.imread(image_path)
    if img is None: return
    faces = app.get(img)
    if len(faces) > 0:
        # 取检测到的第一张脸的特征
        face_db[name] = faces[0].normed_embedding
        print(f"成功注册: {name}")

# 注册示例 (假设 img2 是孙佳美)
if 'faces2' in locals() and len(faces2) > 0:
    face_db['孙佳美'] = faces2[0].normed_embedding
    print("已从之前的运行结果中注册：孙佳美")

# 2. 定义比对函数
def identify_face(target_embedding, threshold=0.5):
    best_name = "Unknown"
    max_sim = 0

    for name, db_embedding in face_db.items():
        # 计算余弦相似度
        sim = np.dot(target_embedding, db_embedding)
        if sim > max_sim:
            max_sim = sim
            best_name = name

    if max_sim < threshold:
        return "Unknown", max_sim
    return best_name, max_sim

# 3. 测试识别
if 'faces' in locals():
    for i, face in enumerate(faces):
        name, score = identify_face(face.normed_embedding)
        print(f"人脸 {i}: 识别结果 = {name}, 相似度分数 = {score:.4f}")

### 使用 FAISS 加速向量检索
当底库变大时，我们可以将 `face_db` 中的向量构建成一个 FAISS 索引。这样搜索时间会从 $O(N)$ 降到几乎恒定的时间。

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss

# 1. 准备数据
names = list(face_db.keys())
embeddings = np.array(list(face_db.values())).astype('float32')

# 2. 创建 FAISS 索引 (使用内积索引 IndexFlatIP，因为我们的向量已归一化，内积等同于余弦相似度)
dimension = embeddings.shape[1]  # 512
index = faiss.IndexFlatIP(dimension)

# 3. 将底库向量添加到索引中
index.add(embeddings)

print(f"FAISS 索引构建完成，底库包含 {index.ntotal} 个特征向量。")

def search_faiss(query_embedding, top_k=1):
    # FAISS 搜索输入需要是 (1, 512) 的矩阵
    query_embedding = query_embedding.reshape(1, -1).astype('float32')

    # 搜索最相似的 k 个结果
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for i in range(top_k):
        idx = indices[0][i]
        if idx != -1: # -1 表示未找到足够结果
            results.append({
                'name': names[idx],
                'similarity': float(distances[0][i])
            })
    return results

# 4. 测试搜索速度
if 'target_embedding' in locals():
    match = search_faiss(target_embedding, top_k=1)[0]
    print(f"FAISS 检索结果: {match['name']}, 相似度: {match['similarity']:.4f}")

# 创建gradio界面

## 使用 index 作为内存数据库
支持界面上传人脸以及拍照，以及人脸识别

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import numpy as np
import cv2

def gradio_register(name, image):
    if image is None or name == "":
        return "请输入姓名并上传图片"

    # Gradio 传入的是 RGB，InsightFace 通常需要 BGR
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    faces = app.get(img_bgr)

    if len(faces) == 0:
        return "未检测到人脸，请重试"

    # 提取特征并归一化
    embedding = faces[0].normed_embedding.astype('float32')

    # 更新本地 face_db
    face_db[name] = embedding

    # 更新 FAISS 索引
    global index, names
    names.append(name)
    index.add(embedding.reshape(1, -1))

    print(f"[系统日志] 用户上传并注册了新人员: {name}")
    return f"成功注册: {name} (当前底库人数: {len(names)})"

def gradio_identify(image):
    if image is None:
        return "请上传图片", None

    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    faces = app.get(img_bgr)

    if len(faces) == 0:
        return "未检测到人脸", None

    # 获取第一个检测到的人脸进行检索
    query_vec = faces[0].normed_embedding
    match_results = search_faiss(query_vec, top_k=1)

    if not match_results:
        return "底库为空", None

    best_match = match_results[0]
    # 绘制结果图片
    res_img = app.draw_on(img_bgr, [faces[0]])
    res_img_rgb = cv2.cvtColor(res_img, cv2.COLOR_BGR2RGB)

    result_text = f"识别结果: {best_match['name']}\n相似度: {best_match['similarity']:.4f}"
    return result_text, res_img_rgb

# 构建界面
with gr.Blocks() as demo:
    gr.Markdown("## 🚀 InsightFace + FAISS 人脸管理系统")

    with gr.Tab("人脸注册"):
        with gr.Row():
            reg_input_name = gr.Textbox(label="人员姓名")
            reg_input_img = gr.Image(label="上传人脸照片")
        reg_btn = gr.Button("提交注册")
        reg_output = gr.Textbox(label="状态")
        reg_btn.click(gradio_register, inputs=[reg_input_name, reg_input_img], outputs=reg_output)

    with gr.Tab("人脸识别"):
        with gr.Row():
            ident_input_img = gr.Image(label="上传待识别照片")
            ident_output_img = gr.Image(label="检测结果")
        ident_output_text = gr.Textbox(label="识别信息")
        ident_btn = gr.Button("开始识别")
        ident_btn.click(gradio_identify, inputs=ident_input_img, outputs=[ident_output_text, ident_output_img])

demo.launch(share=True, debug=True)

## 使用 supabase 作为数据库

⚠️ **安全提示**：请使用环境变量存储 Supabase 凭证，而不是硬编码在代码中。

在 Google Colab 中，你可以使用：
```python
from google.colab import userdata
supabase_url = userdata.get('SUPABASE_URL')
supabase_key = userdata.get('SUPABASE_KEY')
```

In [ ]:
!pip install -q supabase

In [ ]:
from supabase import create_client, Client
import os

# 从环境变量获取凭证
SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    print("⚠️  请设置 SUPABASE_URL 和 SUPABASE_KEY 环境变量")
    print("在 Google Colab 中，使用: userdata.get('SUPABASE_URL') 和 userdata.get('SUPABASE_KEY')")
else:
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("Supabase 客户端初始化成功！")

### ⚠️ 重要步骤：在 Supabase SQL Editor 中执行

由于 Python SDK 无法直接修改数据库 Schema，请登录你的 Supabase 后台，打开 **SQL Editor**，运行以下命令来创建支持向量搜索的表：

```sql
-- 1. 开启向量扩展
create extension if not exists vector;

-- 2. 创建人脸底库表
create table if not exists face_profiles (
  id bigserial primary key,
  name text not null,
  embedding vector(512) -- InsightFace buffalo_l 输出的是 512 维向量
);

-- 3. 创建相似度搜索函数 (供 API 调用)
create or replace function match_faces (
  query_embedding vector(512),
  match_threshold float,
  match_count int
)
returns table (
  id bigint,
  name text,
  similarity float
)
language plpgsql
as $$
begin
  return query
  select
    face_profiles.id,
    face_profiles.name,
    1 - (face_profiles.embedding <=> query_embedding) as similarity
  from face_profiles
  where 1 - (face_profiles.embedding <=> query_embedding) > match_threshold
  order by face_profiles.embedding <=> query_embedding
  limit match_count;
end;
$$;
```

In [ ]:
def sync_local_to_supabase():
    """将本地 face_db 的数据批量同步到 Supabase"""
    if not supabase:
        print("Supabase 客户端未初始化")
        return
    
    data_to_insert = []
    for name, embedding in face_db.items():
        # 转换 embedding 为 list 格式以适配 JSON
        data_to_insert.append({
            "name": name,
            "embedding": embedding.tolist()
        })

    if data_to_insert:
        try:
            response = supabase.table("face_profiles").upsert(data_to_insert).execute()
            print(f"成功同步 {len(data_to_insert)} 条记录到 Supabase")
        except Exception as e:
            print(f"同步失败: {e}")
    else:
        print("本地底库为空，无需同步")

# 执行同步
# sync_local_to_supabase()